In [1]:
import pandas as pd
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")

dfs = []

for file in RAW_DATA_DIR.glob("*.csv"):
    print("Loading:", file.name)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

merged_df = pd.concat(dfs, axis=0, ignore_index=True)

print("Merged dataset shape:", merged_df.shape)


Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Merged dataset shape: (2830743, 79)


In [4]:
print("Columns count:", len(merged_df.columns))
print("Label distribution (top 20):")
print(merged_df[" Label"].value_counts().head(20))


Columns count: 79
Label distribution (top 20):
 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [5]:
# Strip leading/trailing spaces from column names
merged_df.columns = merged_df.columns.str.strip()

# Verify
print(merged_df.columns.tolist())


['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [6]:
print("Label column exists:", "Label" in merged_df.columns)


Label column exists: True


In [7]:
print("Column count after strip:", len(merged_df.columns))
print("Duplicate columns:", merged_df.columns.duplicated().sum())


Column count after strip: 79
Duplicate columns: 0


In [8]:
import numpy as np

# Replace positive and negative infinity with NaN
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Verify infinity removal
print("Remaining infinity values:",
      np.isinf(merged_df.select_dtypes(include=[int, float])).sum().sum())


Remaining infinity values: 0


In [9]:
nan_before = merged_df.isna().sum().sum()
print("Total NaN values before imputation:", nan_before)


Total NaN values before imputation: 5734


In [10]:
numeric_cols = merged_df.select_dtypes(include=[int, float]).columns

merged_df[numeric_cols] = merged_df[numeric_cols].fillna(
    merged_df[numeric_cols].median()
)


In [11]:
print("NaN values after imputation:",
      merged_df.isna().sum().sum())

print("Infinity values after imputation:",
      np.isinf(merged_df.select_dtypes(include=[int, float])).sum().sum())


NaN values after imputation: 0
Infinity values after imputation: 0


In [12]:
# Clean label text
merged_df["Label"] = (
    merged_df["Label"]
    .str.replace("�", "-", regex=False)
    .str.strip()
)


In [13]:
label_mapping = {
    "BENIGN": "BENIGN",

    # DoS family
    "DoS Hulk": "DoS",
    "DoS GoldenEye": "DoS",
    "DoS slowloris": "DoS",
    "DoS Slowhttptest": "DoS",

    # DDoS
    "DDoS": "DDoS",

    # PortScan
    "PortScan": "PortScan",

    # Bot
    "Bot": "Bot",

    # Brute Force
    "FTP-Patator": "BruteForce",
    "SSH-Patator": "BruteForce",

    # Web Attacks
    "Web Attack - Brute Force": "WebAttack",
    "Web Attack - XSS": "WebAttack",
    "Web Attack - Sql Injection": "WebAttack",

    # Others
    "Infiltration": "Infiltration",
    "Heartbleed": "Heartbleed"
}


In [14]:
merged_df["Attack_Type"] = merged_df["Label"].map(label_mapping)


In [15]:
print("Unmapped labels:",
      merged_df[merged_df["Attack_Type"].isna()]["Label"].unique())

print("\nFinal class distribution:")
print(merged_df["Attack_Type"].value_counts())


Unmapped labels: []

Final class distribution:
Attack_Type
BENIGN          2273097
DoS              252661
PortScan         158930
DDoS             128027
BruteForce        13835
WebAttack          2180
Bot                1966
Infiltration         36
Heartbleed           11
Name: count, dtype: int64


In [16]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
merged_df["Attack_Label"] = label_encoder.fit_transform(
    merged_df["Attack_Type"]
)

print("Label encoding mapping:")
for cls, idx in zip(label_encoder.classes_,
                    label_encoder.transform(label_encoder.classes_)):
    print(f"{cls} -> {idx}")


Label encoding mapping:
BENIGN -> 0
Bot -> 1
BruteForce -> 2
DDoS -> 3
DoS -> 4
Heartbleed -> 5
Infiltration -> 6
PortScan -> 7
WebAttack -> 8


In [17]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [20]:
#Save the Cleaned Dataset
final_df = merged_df.drop(columns=["Label"])

final_path = PROCESSED_DIR / "cicids2017_processed.csv"
final_df.to_csv(final_path, index=False)

print("Saved processed dataset to:", final_path)
print("Final shape:", final_df.shape)


Saved processed dataset to: ..\data\processed\cicids2017_processed.csv
Final shape: (2830743, 80)


In [21]:
#Save Feature List
import json

feature_cols = [col for col in final_df.columns
                if col not in ["Attack_Label", "Attack_Type"]]

features_path = Path("../features/feature_list.json")
features_path.parent.mkdir(exist_ok=True)

with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

print("Saved feature list:", features_path)
print("Total features:", len(feature_cols))


Saved feature list: ..\features\feature_list.json
Total features: 78


In [22]:
import joblib

encoder_path = Path("../models/label_encoder.pkl")
encoder_path.parent.mkdir(exist_ok=True)

joblib.dump(label_encoder, encoder_path)

print("Saved label encoder to:", encoder_path)


Saved label encoder to: ..\models\label_encoder.pkl


In [23]:
label_map_path = Path("../models/label_mapping.json")

with open(label_map_path, "w") as f:
    json.dump(
        {cls: int(label_encoder.transform([cls])[0])
         for cls in label_encoder.classes_},
        f,
        indent=2
    )

print("Saved label mapping to:", label_map_path)


Saved label mapping to: ..\models\label_mapping.json


In [24]:
print(final_df.isna().sum().sum(), "NaN values")
print(final_df.select_dtypes(include=[int, float]).shape)


0 NaN values
(2830743, 79)
